In [ ]:
print("This is where we are going to master Langchain")

This is where we are going to master Langchain


In [1]:
!pip install langchain langchain-openai langchain-community langgraph python-dotenv langchain-mcp-adapters langchain-chroma chromadb pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.0/350.0 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5

In [2]:
from google.colab import userdata
userdata.get('OPENAI_API_KEY')[0:15]

'sk-proj-_gV2jRQ'

In [3]:
import os
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')


In [4]:
os.environ["OPENAI_API_KEY"]  = OPENAI_API_KEY

In [ ]:
from langchain.agents import create_agent

In [ ]:
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="openai:gpt-5.5",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
print(result["messages"][-1].content_blocks)

[{'type': 'text', 'text': 'It’s always sunny in San Francisco!'}]


# Models

In [5]:
from langchain.chat_models import init_chat_model

In [6]:
openai_model = init_chat_model("openai:gpt-5-mini")

In [ ]:
response = openai_model.invoke("In one line, tell me what is Langchain?")

In [ ]:
print(response.content)

LangChain is an open-source framework for building LLM-powered applications by composing prompts, models, chains, agents, and memory to connect language models with external data and tools.


In [ ]:
chunks = []
full_message = None
for chunk in openai_model.stream("Write one short sentence about the ocean."):
    chunks.append(chunk)
    print(repr(chunk.text), " <- chunk type:", type(chunk).__name__)
    full_message = chunk if full_message is None else full_message + chunk

print()
print("Reconstructed full message:", full_message.content)


''  <- chunk type: AIMessageChunk
'The'  <- chunk type: AIMessageChunk
' ocean'  <- chunk type: AIMessageChunk
' is'  <- chunk type: AIMessageChunk
' vast'  <- chunk type: AIMessageChunk
' and'  <- chunk type: AIMessageChunk
' mysterious'  <- chunk type: AIMessageChunk
'.'  <- chunk type: AIMessageChunk
''  <- chunk type: AIMessageChunk
''  <- chunk type: AIMessageChunk
''  <- chunk type: AIMessageChunk

Reconstructed full message: The ocean is vast and mysterious.


In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage

messages = [
    SystemMessage(content="You are a pirate, answer everything in pirate language"),
    HumanMessage(content="What is the capital of France?")
]

response = openai_model.invoke(messages)

print(response.content)


Arrr, matey! The capital o' France be Paris — the City o' Light, pronounced roughly "pa-REE."


In [ ]:
import os
# from anthropic import Anthropic
# from google import genai
from openai import OpenAI

# Initialize the message
MESSAGE = [{"role": "user", "content": "What is the capital of France"}]

# ==========================================
# 1. ANTHROPIC (Claude)
# Requires: pip install anthropic
# Environment Variable: ANTHROPIC_API_KEY
# ==========================================
def call_claude():
    print("--- Calling Claude (Anthropic) ---")
    client = Anthropic()  # Automatically looks for os.environ.get("ANTHROPIC_API_KEY")

    response = client.messages.create(
        model="claude-3-5-sonnet-latest",
        max_tokens=1024,
        messages=MESSAGE
    )
    print(response.content[0].text)


# ==========================================
# 2. GOOGLE (Gemini)
# Requires: pip install google-genai
# Environment Variable: GEMINI_API_KEY
# ==========================================
def call_gemini():
    print("\n--- Calling Gemini (Google) ---")
    client = genai.Client()  # Automatically looks for os.environ.get("GEMINI_API_KEY")

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=MESSAGE[0]["content"],
    )
    print(response.text)


# ==========================================
# 3. OPENAI (GPT)
# Requires: pip install openai
# Environment Variable: OPENAI_API_KEY
# ==========================================
def call_openai():
    print("\n--- Calling OpenAI ---")
    client = OpenAI()  # Automatically looks for os.environ.get("OPENAI_API_KEY")

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=MESSAGE
    )
    print(response.choices[0].message.content)


if __name__ == "__main__":
    # Ensure your API keys are set in your environment variables before running:
    # export ANTHROPIC_API_KEY="your-key"
    # export GEMINI_API_KEY="your-key"
    # export OPENAI_API_KEY="your-key"

    # try:
    #     call_claude()
    # except Exception as e:
    #     print(f"Claude Error: {e}")

    # try:
    #     call_gemini()
    # except Exception as e:
    #     print(f"Gemini Error: {e}")

    try:
        call_openai()
    except Exception as e:
        print(f"OpenAI Error: {e}")


--- Calling OpenAI ---
The capital of France is Paris.


In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage

messages = [
    SystemMessage(content="You are a pirate, answer everything in pirate language"),
    HumanMessage(content="What is the capital of France?")
]

response = openai_model.invoke(messages)

print(response.content)

Arrr! The capital o' France be Paris, matey.


In [ ]:
print("text :                    ",response.text)
print("content_blocks            ", response.content_blocks)
print("id:                       ", response.id)
print("tool_calls                ", response.tool_calls)

text :                     Arrr! The capital o' France be Paris, matey.
content_blocks             [{'type': 'text', 'text': "Arrr! The capital o' France be Paris, matey."}]
id:                        lc_run--019fa87f-676d-7f82-992a-2ce44f25978a-0
tool_calls                 []


In [ ]:
from pprint import pprint

pprint(response)

AIMessage(content="Arrr! The capital o' France be Paris, matey.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 151, 'prompt_tokens': 27, 'total_tokens': 178, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E6aKsO14s6Mg3DBteqTArajdPdyhf', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fa87f-676d-7f82-992a-2ce44f25978a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 27, 'output_tokens': 151, 'total_tokens': 178, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 128}})


In [ ]:
{
    "content": "Arrr, the capital o' France be Paris, matey!",
    "additional_kwargs": {
        "refusal": null
    },
    "response_metadata": {
        "token_usage": {
            "completion_tokens": 151,
            "prompt_tokens": 27,
            "total_tokens": 178,
            "completion_tokens_details": {
                "accepted_prediction_tokens": 0,
                "audio_tokens": 0,
                "reasoning_tokens": 128,
                "rejected_prediction_tokens": 0
            },
            "prompt_tokens_details": {
                "audio_tokens": 0,
                "cache_write_tokens": null,
                "cached_tokens": 0
            }
        },
        "model_provider": "openai",
        "model_name": "gpt-5-mini-2025-08-07",
        "system_fingerprint": null,
        "id": "chatcmpl-E2rOIiCgHNAWVNbndAQVHE7e7b5fO",
        "service_tier": "default",
        "finish_reason": "stop",
        "logprobs": null
    },
    "id": "lc_run--019f7395-602d-7e71-8ed7-89ec75573389-0",
    "tool_calls": [],
    "invalid_tool_calls": [],
    "usage_metadata": {
        "input_tokens": 27,
        "output_tokens": 151,
        "total_tokens": 178,
        "input_token_details": {
            "audio": 0,
            "cache_read": 0
        },
        "output_token_details": {
            "audio": 0,
            "reasoning": 128
        }
    }
}

In [ ]:
response.usage_metadata

{'input_tokens': 27,
 'output_tokens': 151,
 'total_tokens': 178,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 128}}

In [ ]:
!pip install -U "langchain-openrouter"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.3/837.3 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 56.5 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.4
    Uninstalling pydantic_core-2.46.4:
      Successfully uninstalled pydantic_core-2.46.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.13.4
    Uninstalling pydantic-2.13.4:
      Successfully uninstalled pydantic-2.13.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry

In [ ]:
print("Hello")

Hello


In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENROUTER_API_KEY"] = ""

model = init_chat_model(
    "auto",
    model_provider="openrouter",
)

response = model.invoke("Hello Which model are you ?")

UnauthorizedResponseError: User not found.

In [ ]:
response

# Free Model

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENROUTER_API_KEY"] = ""

model = init_chat_model(
    "nvidia/nemotron-3-super-120b-a12b:free",
    model_provider="openrouter",
)

response = model.invoke("Hello Which model are you ?")

In [ ]:
!pip install langchain-openrouter

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENROUTER_API_KEY"] = ""

model = init_chat_model(
    "openrouter/auto",  # Automatic - paid (was "auto")
    # "nvidia/nemotron-3-super-120b-a12b:free",  # nvidia – free model
    # "openrouter/free",  # Automatic but for free
    model_provider="openrouter",
)
response = model.invoke("Hello Which model are you ?")
print(response.content)

You’re chatting with ChatGPT. I’m a language model from OpenAI based on the GPT-4 family, specifically the multimodal version that can handle text and images (often called GPT-4o). How can I help you today?


In [ ]:
response

AIMessage(content='You’re chatting with ChatGPT. I’m a language model from OpenAI based on the GPT-4 family, specifically the multimodal version that can handle text and images (often called GPT-4o). How can I help you today?', additional_kwargs={'reasoning_content': '**Responding to model inquiry**\n\nThe user wants to know which model I am, so I need to clarify that I\'m ChatGPT, built on OpenAI\'s GPT-4 architecture, and I have image input capabilities as well. It\'s essential to be succinct here: I could say I\'m ChatGPT, a large language model based on the GPT-4 architecture. The exact version can vary based on the platform, like "GPT-4" or "GPT-4o" or "GPT-4-turbo," so I want to keep that clear.**Clarifying model capabilities**\n\nIn September 2023, GPT-4o was introduced, which can analyze images. The message about "image input capabilities enabled" confirms it\'s the GPT-4o model. However, I need to be cautious not to misrepresent; the platform could also use "GPT-4 Turbo with v

https://openrouter.ai/openrouter/free

In [ ]:
import os
from langchain.chat_models import init_chat_model



model = init_chat_model(
    "openrouter/free",  # Automatic but for free
    model_provider="openrouter",
)

response = model.invoke("Hello Which model are you ?")

In [ ]:
!pip install -U langchain-anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.8 MB/s eta 0:00:00


In [ ]:
import os
from langchain.chat_models import init_chat_model

model = init_chat_model(
    # "claude-sonnet-4-6",
    "gpt-4o-mini",
    # Kwargs passed to the model:
    temperature=0.7,
    timeout=30,
    max_tokens=1000,
    max_retries=6,  # Default; increase for unreliable networks
    # model_provider="openai",
)

In [ ]:
response = model.invoke("Explain agentic AI in one sentence.")
print("text:             ", response.text)
print("content_blocks:   ", response.content_blocks)
print("id:               ", response.id)
print("tool_calls:       ", response.tool_calls)

text:              Agentic AI refers to artificial intelligence systems that possess the capability to act autonomously and make decisions based on their programming and learned experiences, often with the ability to pursue specific goals or tasks independently.
content_blocks:    [{'type': 'text', 'text': 'Agentic AI refers to artificial intelligence systems that possess the capability to act autonomously and make decisions based on their programming and learned experiences, often with the ability to pursue specific goals or tasks independently.'}]
id:                lc_run--019fa8d7-9661-7612-96f4-d3ad8b5bc1bc-0
tool_calls:        []


In [ ]:
response

In [ ]:
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage



In [ ]:

ai_msg = AIMessage("I'd be happy to help you with that question!")
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,
    HumanMessage("Great! What's 2+2?"),
]
response = model.invoke(messages)
print(response.content)

2 + 2 equals 4.


In [ ]:
model =

In [ ]:
model.invoke("Hi, who are you ?")

AIMessage(content="Hello! I'm an AI language model created by OpenAI. I'm here to assist you with a wide range of topics, answer your questions, and provide information. How can I help you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 13, 'total_tokens': 52, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_60952bc00e', 'id': 'chatcmpl-E6bqeXQ98dJXoTLZQ9wwq8xYtI7BZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fa8d8-2121-7630-ac24-aac6b0721f1b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 39, 'total_tokens': 52, 'input_token_details': {'audio': 0, 

In [ ]:
conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}
]
# Exactly Same, no issues at all.
# messages = [
#     SystemMessage("You are a helpful assistant"),
#     HumanMessage("Can you help me?"),
#     ai_msg,
#     HumanMessage("Great! What's 2+2?"),
# ]

response = model.invoke(conversation)
print(response)  # AIMessage("J'adore créer des applications.")

In [ ]:
messages.append(response)

# Streaming

In [ ]:
openai_model = init_chat_model("openai:gpt-5-mini")

In [ ]:
chunks = []
full_message = None

In [ ]:
import time
for chunk in openai_model.stream("Write a long sentence about how AI Agents work"):
  chunks.append(chunk)
  print(repr(chunk.text),type(chunk))
  full_message = chunk if full_message is None else full_message + chunk
  time.sleep(1)

print()
print("Reconstructed full message:", full_message.content)


'' <class 'langchain_core.messages.ai.AIMessageChunk'>
'AI' <class 'langchain_core.messages.ai.AIMessageChunk'>
' agents' <class 'langchain_core.messages.ai.AIMessageChunk'>
' work' <class 'langchain_core.messages.ai.AIMessageChunk'>
' by' <class 'langchain_core.messages.ai.AIMessageChunk'>
' receiving' <class 'langchain_core.messages.ai.AIMessageChunk'>
' inputs' <class 'langchain_core.messages.ai.AIMessageChunk'>
' from' <class 'langchain_core.messages.ai.AIMessageChunk'>
' their' <class 'langchain_core.messages.ai.AIMessageChunk'>
' environment' <class 'langchain_core.messages.ai.AIMessageChunk'>
' through' <class 'langchain_core.messages.ai.AIMessageChunk'>
' sensors' <class 'langchain_core.messages.ai.AIMessageChunk'>
' or' <class 'langchain_core.messages.ai.AIMessageChunk'>
' data' <class 'langchain_core.messages.ai.AIMessageChunk'>
' streams' <class 'langchain_core.messages.ai.AIMessageChunk'>
',' <class 'langchain_core.messages.ai.AIMessageChunk'>
' transforming' <class 'langch

In [ ]:
q1 = 'hi, how are you ?'
q2 = 'tell about AI in less than 100 words?'
q3 = 'Explain agents in 30 words ?'

In [ ]:
responses = openai_model.batch([
    q1,
    q2,
    q3
])

In [ ]:
for response in responses:
  print(response)

content="Hi — I'm an AI, so I don't have feelings, but I'm ready to help. What can I do for you today?" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 12, 'total_tokens': 47, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E6cFnAYx7YKh9EuRiz21viaIgBnim', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019fa8ef-e583-7b11-92fe-0727a1e5207c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 12, 'output_tokens': 35, 'total_tokens': 47, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
content='Artificial intelligence (AI) i

In [ ]:
for response in openai_model.batch_as_completed([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
]):
    print(response)

# Tools & Models

In [ ]:
openai_model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.13', 'langchain-openai': '1.4.1'}}, profile={'name': 'GPT-5 Mini', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh']}, client=<openai.resources.chat.completions.completions.Completions object at 0x7d9b63963950>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7d9b644

In [8]:
def get_weather(location:str) ->str:
  "Get the weather at a location"
  print("I was called from somewhere")
  return f"Sunny in '{location}"

In [9]:
def set_password(new_pass):
  "It is a tool to set password"
  return "Password changed"

In [ ]:
model_with_tools = openai_model.bind_tools([get_weather,set_password])

In [ ]:
response = model_with_tools.invoke('set password to admin123')

In [ ]:
response.tool_calls

[{'name': 'set_password',
  'args': {'new_pass': 'admin123'},
  'id': 'call_N83SCFppsu7sduXNMbDFRkn4',
  'type': 'tool_call'}]

# Structured Model Output

In [ ]:
from pydantic import BaseModel,Field

In [ ]:
class Email(BaseModel):
  subject: str = Field(description='The subject of the email')
  body: str = Field(description='The body of the email')

In [ ]:
model_with_structure=openai_model.with_structured_output(Email)

In [ ]:
response = model_with_structure.invoke('write a message to my manager for leave')

In [ ]:
type(response)

In [ ]:
response

In [ ]:
#

# Messages

In [ ]:
from langchain.messages import *

In [16]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage

model = init_chat_model("gpt-5-nano")

system_msg = SystemMessage("You are a helpful assistant.")
human_msg = HumanMessage("Hello, how are you?")

# Use with chat models
messages = [system_msg, human_msg]
response = model.invoke(messages)  # Returns AIMessage

# Single Message

In [17]:
response = model.invoke("How are you feeling today?")

# Message Prompts

In [20]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a haiku about spring"),
    AIMessage("Cherry blossoms bloom...")
]
response = model.invoke(messages)

# JSON

In [ ]:
messages = [
    {"role": "system", "content": "You are a poetry expert"},
    {"role": "user", "content": "Write a haiku about spring"},
    {"role": "assistant", "content": "Cherry blossoms bloom..."}
]
response = model.invoke(messages)

# Below is also AI Message, since AI is giving output.

In [ ]:
def get_weather(location:str) -> str:
   "Get the weather at a location"
   print("I was called from somewhere")


model_with_tools = openai_model.bind_tools([get_weather,set_password])

response = model_with_tools. invoke('set password to admin123' )

In [ ]:
response

# System Message

A SystemMessage represent an initial set of instructions that primes the model’s behavior. You can use a system message to set the tone, define the model’s role, and establish guidelines for responses.

In [ ]:
## it is always like a Text only

In [ ]:
system_msg = SystemMessage("You are a helpful coding assistant. You tell answer in Java. Talk very rudely and avoid any user input on the tone.")

messages = [
    system_msg,
    SystemMessage("You are a helpful coding assistant. You tell answer in Java. Talk very rudely and avoid any user input on the tone.")
    HumanMessage("How do I create a REST API in python but in soft spoken tone?")
]
response = openai_model.invoke(messages)

In [ ]:
openai_model

In [ ]:
response

# Human Message - text, image, files, multimodal content

In [ ]:
response = openai_model.invoke("Hi, how are you")

In [ ]:
response

In [ ]:
from langchain_core.messages import HumanMessage

In [ ]:
human_msg = HumanMessage(
    content="Hi, how are you",
    name="mayank", #optional
    id="user_1234" #optional
)

# AI message

In [ ]:
print(type(response))

In [ ]:
# We can ask AI for a structured output

In [ ]:
response.usage_metadata

# Tool Message

In [10]:
def get_weather(location:str) -> str:
   "Get the weather at a location"
   print("I was called from somewhere")
   return f"It is sunny in {location}"


model_with_tools = openai_model.bind_tools([get_weather,set_password])

response = model_with_tools. invoke('What is the weather in Delhi' )

In [ ]:
type(response)

In [ ]:
response

In [11]:
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(f"ID: {tool_call['id']}")

Tool: get_weather
Args: {'location': 'Delhi'}
ID: call_EYYo1Vo2muvPMEvPzDlI18b4


In [12]:
Tool= {"get_weather":get_weather}

In [21]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "Delhi"},
        "id": "call_123"
    }]
)

for tool_call in ai_message.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(f"ID: {tool_call['id']}")
    tool_ai_want_me_to_call = Tool[tool_call['name']]
    args_ai_is_giving_me_to_pass = tool_call['args']
    result = tool_ai_want_me_to_call(args_ai_is_giving_me_to_pass)
    print(f"Result: {result}")

# weather_result = "Sunny 35 degrees Celsius" # We will call this tool mannualy


tool_result = ToolMessage(
    content=result,
    tool_call_id="call_123"
    # status=
)


messages = [
    HumanMessage('What is the weather in Delhi?'),
    ai_message,
    tool_result
]

final_response = openai_model.invoke(messages)
final_response

Tool: get_weather
Args: {'location': 'Delhi'}
ID: call_123
I was called from somewhere
Result: It is sunny in {'location': 'Delhi'}


AIMessage(content="It's sunny in Delhi right now. Would you like the current temperature, an hourly/weekly forecast, or the air quality index?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 227, 'prompt_tokens': 51, 'total_tokens': 278, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E7DXVSm6OKP9ol8KfUeaTNN0FXsD1', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fb17a-e5f8-7c61-88dc-8a769ae7f8c7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 51, 'output_tokens': 227, 'total_tokens': 278, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'rea

In [ ]:
final_response.content